# LENZ Speed Estimation: Project Walkthrough

This notebook is a concise, report-friendly tour of the current pipeline outputs. It deliberately keeps data loading, feature extraction, modeling, evaluation, and plotting logic inside src/lenz_speed/.

Before opening the results, run the reproducible pipeline from the repository root:

    python run_pipeline.py

## 1. Setup and package API

The setup cell locates the repository root whether the kernel starts from the root or the notebooks/ directory. It then imports the public pipeline functions without reimplementing them.

In [ ]:
from pathlib import Path
import os
import sys
import tempfile

import pandas as pd
from IPython.display import Image, Markdown, display

candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next(
    (path for path in candidates if (path / 'src/lenz_speed').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('Could not locate the repository root.')

os.environ.setdefault(
    'MPLCONFIGDIR',
    str(Path(tempfile.gettempdir()) / 'lenz-speed-matplotlib'),
)
sys.path.insert(0, str(REPO_ROOT / 'src'))

from lenz_speed import (
    build_windowed_feature_table,
    feature_ablation,
    generate_all_plots,
    same_subject_cadence_stress_test,
    same_subject_standard_validation,
)

print(f'Repository root: {REPO_ROOT}')
print('Imported the LENZ pipeline API successfully.')

## 2. Pipeline artifacts

The notebook reads the artifacts created by run_pipeline.py. This keeps the walkthrough fast and ensures the displayed results match the saved report tables and figures.

In [ ]:
artifact_paths = {
    'features': REPO_ROOT / 'data/processed/windowed_features.csv',
    'standard_metrics': REPO_ROOT / 'outputs/tables/same_subject_standard_metrics.csv',
    'standard_predictions': REPO_ROOT / 'outputs/tables/same_subject_standard_predictions.csv',
    'stress_metrics': REPO_ROOT / 'outputs/tables/cadence_stress_metrics.csv',
    'ablation_metrics': REPO_ROOT / 'outputs/tables/feature_ablation_metrics.csv',
}
missing = [path for path in artifact_paths.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(
        'Missing pipeline artifacts. Run python run_pipeline.py first: '
        + ', '.join(str(path) for path in missing)
    )

features = pd.read_csv(artifact_paths['features'])
standard_metrics = pd.read_csv(artifact_paths['standard_metrics'])
stress_metrics = pd.read_csv(artifact_paths['stress_metrics'])
ablation_metrics = pd.read_csv(artifact_paths['ablation_metrics'])
print('All required pipeline artifacts are available.')

## 3. Dataset overview

Each row in the processed table represents one complete five-second IMU window. The overview below shows the analysis scale after manifest exclusions, trimming, and signal validation.

In [ ]:
dataset_overview = pd.DataFrame(
    {
        'measure': [
            'Window rows',
            'Recordings',
            'Subjects',
            'Minimum speed (mph)',
            'Maximum speed (mph)',
        ],
        'value': [
            len(features),
            features['recording_id'].nunique(),
            features['subject_id'].nunique(),
            features['speed_mph'].min(),
            features['speed_mph'].max(),
        ],
    }
)
display(dataset_overview)

## 4. Recordings by session

Session-level counts clarify which recordings support training, standard validation, cadence manipulation, and cross-subject analysis.

In [ ]:
recordings_by_session = (
    features.groupby(['subject_id', 'session'], as_index=False)
    .agg(
        recordings=('recording_id', 'nunique'),
        windows=('window_index', 'size'),
        minimum_speed_mph=('speed_mph', 'min'),
        maximum_speed_mph=('speed_mph', 'max'),
    )
)
display(recordings_by_session)

## 5. Standard same-subject validation

The headline validation trains on Subject 1 Day 3 and tests on approved Day 2 recordings plus normal-cadence Day 4 recordings. Cadence-manipulation trials are intentionally excluded here.

In [ ]:
standard_report = standard_metrics[
    ['model', 'n_train_windows', 'n_test_windows', 'MAE', 'RMSE', 'R2']
].sort_values('MAE')
display(standard_report.round(4))

## 6. Cadence-manipulation stress test

This separate test trains on Day 3 but evaluates every Subject 1 Day 4 cadence condition. It measures robustness when cadence is deliberately changed and should not be treated as the headline validation score.

In [ ]:
stress_report = stress_metrics[
    ['model', 'n_train_windows', 'n_test_windows', 'MAE', 'RMSE', 'R2']
].sort_values('MAE')
display(stress_report.round(4))

## 7. Feature ablation

The ablation table compares compact feature sets using Linear Regression and Random Forest on the standard validation split. Lower MAE indicates better speed estimation.

In [ ]:
ablation_report = ablation_metrics[
    ['feature_set', 'features', 'model', 'MAE', 'RMSE', 'R2']
].sort_values(['MAE', 'feature_set'])
display(ablation_report.round(4))

## 8. Generated figures

These report-ready figures are generated by lenz_speed.plotting during run_pipeline.py. The notebook embeds the saved PNG files rather than reproducing plotting logic.

In [ ]:
figure_names = [
    'predicted_vs_actual_standard.png',
    'residual_by_speed_standard.png',
    'random_forest_prediction_trace_standard.png',
    'cadence_stress_predicted_vs_actual.png',
    'feature_ablation_mae.png',
]
figure_dir = REPO_ROOT / 'outputs/figures'
for figure_name in figure_names:
    figure_path = figure_dir / figure_name
    if not figure_path.is_file():
        raise FileNotFoundError(
            f'Missing figure {figure_path}. Run python run_pipeline.py first.'
        )
    title = figure_name.removesuffix('.png').replace('_', ' ').title()
    display(Markdown(f'### {title}'))
    display(Image(filename=str(figure_path), width=1000))

## 9. Takeaways for the capstone report

- Report standard validation and cadence stress results separately.
- Use the standard Random Forest result as the primary same-subject accuracy result.
- Use the cadence stress test to discuss failure modes and cadence dependence.
- Use the ablation table to support claims about which inertial features add information beyond cadence.

For updated results, rerun python run_pipeline.py and then restart and run this notebook from the top.